# 저장된 gold 예측 CPU 재집계
**일반 CPU 런타임 → GITHUB_TOKEN 허용 → 모두 실행 → Drive 승인 → ZIP 전달. GPU는 필요 없습니다.**

PR56 v1 및 수정 v2의 저장된 QA/예측값에서 Q1·Q2 누락 지표와 gold 표기를 복구합니다.
원래 예측·표·metadata·추론코드 identity는 수정하지 않고 각 run의 aggregation_v2/에 저장합니다.
응답이 누락되거나 실패/무효면 해당 비교를 제외하고 완료를 인정하지 않습니다. 재집계가 없는 응답을 생성하지는 않습니다.
실행 코드는 792c9eac45eafdedda2013cc31ab969b7df7e041으로 고정하며 fresh/cached 코드의 HEAD와 변경 여부를 검사합니다.


In [ ]:
REPO = "arnold6444/lvlm_insplad"
EXECUTION_COMMIT = "792c9eac45eafdedda2013cc31ab969b7df7e041"
SAVED_RUN_FOLDERS = ["lvlm_insplad_results/gold_v2_q23_candidate_v1",
                     "lvlm_insplad_results/gold_v2_q23_candidate_v2"]


In [ ]:
# Drive 연결 + repo clone + 패키지
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
REPO_DIR = Path("/content/lvlm_q23_reporting_code")

def run(cmd, *, env=None, check=True):
    """Run a command and stream its output into this cell (Colab hides plain subprocess output)."""
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""), flush=True)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-60:]
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed with exit code {p.returncode} (마지막 출력 위 참고)")
    return p.returncode

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT":"1", "GIT_CONFIG_KEY_0":"http.extraHeader",
                "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}", "GIT_TERMINAL_PROMPT":"0"})
# Fetch exactly the locked object, never a moving branch.
if not REPO_DIR.exists():
    REPO_DIR.mkdir(parents=True)
    run(["git", "init", REPO_DIR])
    run(["git", "-C", REPO_DIR, "remote", "add", "origin", f"https://github.com/{REPO}.git"])
    run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", EXECUTION_COMMIT], env=git_env)
    run(["git", "-C", REPO_DIR, "checkout", "--detach", EXECUTION_COMMIT])
else:
    existing_remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True).strip()
    if existing_remote != f"https://github.com/{REPO}.git":
        raise RuntimeError("실행 폴더의 저장소가 다릅니다. 새 런타임을 사용하세요.")
actual_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if actual_revision != EXECUTION_COMMIT:
    raise RuntimeError("고정 커밋과 다른 코드입니다. 새 런타임을 사용하세요.")
if subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO_DIR, text=True).strip():
    raise RuntimeError("고정 코드가 수정됐습니다. 새 런타임을 사용하세요.")
del auth, git_env
run([sys.executable, "-m", "pip", "install", "-q", "pandas>=2.2", "numpy>=1.26", "pillow>=10"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))
from verify_gold_revision import verify as verify_gold_revision
verify_gold_revision(REPO_DIR, EXECUTION_COMMIT)
import pandas as pd


In [ ]:
# 저장된 run 자동 검색. 원본 ZIP/QA 폴더는 없어도 됩니다.
verify_gold_revision(REPO_DIR, EXECUTION_COMMIT)
SAVED_RUNS = [DRIVE_ROOT / folder for folder in SAVED_RUN_FOLDERS
              if (DRIVE_ROOT / folder / "predictions/internvl3_5_8b/steps.jsonl").is_file()]
if not SAVED_RUNS:
    raise FileNotFoundError("저장된 gold 예측이 없습니다. 아직 실행 전이면 gold 평가 노트북을 사용하세요.")
for saved in SAVED_RUNS:
    print("GPU 없이 재집계:", saved)
    run([sys.executable, REPO_DIR / "scripts/reaggregate_gold_v2_q23_candidate.py",
         "--run-dir", saved], check=True)


In [ ]:
for saved in SAVED_RUNS:
    reports = saved / "aggregation_v2"
    status = json.loads((reports / "run_status.json").read_text(encoding="utf-8"))
    meta = json.loads((reports / "aggregation_metadata.json").read_text(encoding="utf-8"))
    assert meta["inference_calls"] == 0
    print("원래 추론 커밋:", meta["source_benchmark_commit"],
          "재집계 커밋:", meta["aggregation_commit"])
    print("전체 완료:", status["complete"], "GPU 추론:", meta["inference_calls"])
    display(pd.DataFrame(status["policies"]))
    if status["complete"]:
        display(pd.read_csv(reports / "leaderboard.csv").round(3))
        display(pd.read_csv(reports / "metrics_by_asset.csv").round(3))
        display(pd.read_csv(reports / "metrics_asset_macro.csv").round(3))
        display(pd.read_csv(reports / "significance.csv").round(3))
    else:
        print("응답 누락/실패/무효가 있어 완료가 아닙니다. 재집계는 추론을 수행하지 않습니다.")
    print("Q3는 gold 영역 유무 보조 점수이며 위치 정확도 null입니다.")


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = Path("/content/gold_q23_reaggregated_results.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for saved in SAVED_RUNS:
        for p in (saved / "aggregation_v2").rglob("*"):
            if p.is_file():
                z.write(p, Path(saved.name) / p.relative_to(saved))
files.download(str(archive))
